# DTL + MLForecast

Decomposed Trend-Local forecasting for non-seasonal panel time series. The DTL wrapper extracts a robust LOWESS trend, forecasts it with StatsForecast, and models the residual component with MLForecast.

In [1]:
import os
import sys
sys.path.append(os.path.abspath('../..'))

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from mlforecast import MLForecast
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, rmse

from tinyshift.modelling import DTLWrapper
from tinyshift.series import fva_rmae

## Build a non-seasonal panel

The example uses deterministic data so the notebook can run without downloading an external dataset.

In [2]:
rng = np.random.RandomState(42)
dates = pd.date_range('2018-01-01', periods=96, freq='MS')
rows = []
for unique_id, offset, slope in [('series_a', 20.0, 0.30), ('series_b', 35.0, -0.10)]:
    t = np.arange(len(dates), dtype=float)
    trend = offset + slope * t + 0.015 * t ** 2
    residual = 2.5 * np.sin(t / 2.5) + rng.normal(scale=0.7, size=len(t))
    rows.extend(zip([unique_id] * len(dates), dates, trend + residual))

df = pd.DataFrame(rows, columns=['unique_id', 'ds', 'y'])
df.head()

,unique_id,ds,y
0,series_a,2018-01-01,20.347700
1,series_a,2018-02-01,21.191761
2,series_a,2018-03-01,22.906772
3,series_a,2018-04-01,24.431219
4,series_a,2018-05-01,23.775027


In [3]:
horizon = 12
train = df.groupby('unique_id', group_keys=False).apply(lambda group: group.iloc[:-horizon]).reset_index(drop=True)
test = df.groupby('unique_id', group_keys=False).apply(lambda group: group.iloc[-horizon:]).reset_index(drop=True)

train.groupby('unique_id').size(), test.groupby('unique_id').size()

/tmp/ipykernel_6296/2430235546.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train = df.groupby('unique_id', group_keys=False).apply(lambda group: group.iloc[:-horizon]).reset_index(drop=True)
/tmp/ipykernel_6296/2430235546.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test = df.groupby('unique_id', group_keys=False).apply(lambda group: group.iloc[-horizon:]).reset_index(drop=True)


(unique_id
 series_a    84
 series_b    84
 dtype: int64,
 unique_id
 series_a    12
 series_b    12
 dtype: int64)

## Configure the residual forecaster

`mf_resid` is the unfitted residual-model template. DTL creates an isolated fitted copy for each `unique_id`.

In [4]:
residual_forecaster = MLForecast(
    models=[
        LinearRegression(),
        RandomForestRegressor(n_estimators=100, random_state=42),
    ],
    freq='MS',
    lags=[1, 2, 3, 6],
)

dtl = DTLWrapper(
    mf_resid=residual_forecaster,
    trend_frac=0.2,
    robust=True,
)

In [5]:
dtl.fit(train, id_col='unique_id', time_col='ds', target_col='y')

forecast = dtl.predict(h=horizon)
forecast.head()

/home/heylucasleao/tinyshift/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,unique_id,ds,LinearRegression,RandomForestRegressor
0,series_a,2025-01-01,153.765229,154.242754
1,series_a,2025-02-01,157.032532,157.697830
2,series_a,2025-03-01,159.308791,159.454566
3,series_a,2025-04-01,161.766760,161.895088
4,series_a,2025-05-01,164.100848,164.242101


## Evaluate the forecasts

The wrapper preserves one prediction column per residual estimator.

In [6]:
actual = test[['unique_id', 'ds', 'y']]
result = forecast.merge(actual, on=['unique_id', 'ds'], how='left')

metrics = evaluate(
    result,
    metrics=[rmse, mae],
    models=['LinearRegression', 'RandomForestRegressor'],
    id_col='unique_id',
    time_col='ds',
    target_col='y',
)
metrics

,unique_id,metric,LinearRegression,RandomForestRegressor
0,series_a,rmse,3.021105,2.809000
1,series_b,rmse,4.789422,4.475031
2,series_a,mae,2.873324,2.744112
3,series_b,mae,4.553005,4.256120


## Horizontal stabilization

Stabilization can be applied after trend and residual recombination.

In [7]:
stabilized = dtl.predict(
    h=horizon,
    stabilization_method='hfi',
    w_s=0.8,
)[['unique_id', 'ds', 'RandomForestRegressor']]

stabilized.head()

,unique_id,ds,RandomForestRegressor
0,series_a,2025-01-01,154.242754
1,series_a,2025-02-01,154.933769
2,series_a,2025-03-01,155.837928
3,series_a,2025-04-01,157.049360
4,series_a,2025-05-01,158.487908


In [8]:
rf_prediction = result['RandomForestRegressor'].to_numpy()
actual_values = result['y'].to_numpy()
fva_rmae(actual_values, rf_prediction, baseline_type='naive')

0.7687058026377532